## Practise Notebook - PySpark

In [0]:
spark

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
schema = StructType([StructField("data1", StringType()), StructField("value1", StringType())])

In [0]:
df = spark.read.schema(schema)\
            .option("mode", "dropMalformed")\
                .format("csv")\
                .option("header", "false")\
            .load("/Volumes/workspace/default/rawdata/dummy1.csv")
df.show()

In [0]:
df = spark.read.csv('/Volumes/workspace/default/rawdata/dummy1.csv', header=True, inferSchema = True)

# adding indexes
df_with_index = df.withColumn('index', monotonically_increasing_id())

# skipping the lines till headers
# filtered_df = df_with_index.filter(col('index') > 1)

# dropping the index column
# final_df = filtered_df.drop('index')

df_with_index.display()

## Q: Read the df, define schema, filter out empl earning less than 20k, add a col bonus 10% for each emp and calculate total salary after bonus, save final df as parquet

Rahul | Sales | 10000
Bob | Finance | 20000
Jack | Reporting | 15000

In [0]:
schema_sales = StructType([StructField("Employee_Name", StringType()),\
                           StructField("Department", StringType()),\
                             StructField("Salary", IntegerType())])

In [0]:
data = spark.createDataFrame([("Rahul", "Sales", 10000),\
                              ("Bob", "Finance", 20000),\
                              ("Jack", "Reporting", 15000)], schema = schema_sales)

In [0]:
data.display()

In [0]:
data = data.filter(col('Salary') > 10000)
data.display()

In [0]:
data = data.withColumn('Bonus', lit(0.1))

In [0]:
data.display()

In [0]:
# final salary
data = data.withColumn('Updated_Salary', col('Salary')+(col('Salary')*col('Bonus')))
data.display()
                       

In [0]:
data.write.format('parquet').save('/Volumes/workspace/default/rawData/salary.parquet')

## Q: In a below given data replace !$#@ characters with empty space ''

In [0]:
df = spark.createDataFrame([("John", "Smith"), ("Jane", "D!oe"), ("Bob", "Johnson"), ("Mary", "Jones"), ("Mike", "Pant$on"), ( "Tom", "L#ee")], ["First", "Last"])
df.display()

In [0]:
from pyspark.sql.functions import col, regexp_replace

In [0]:
df = df.withColumn('Last', regexp_replace(col('Last'),"[!$#@]", ""))

In [0]:
df.display()

## Q: we have a dataset with values 1,4,5,7,9,10 we need to output missing values

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:

schema = StructType([StructField('ID', IntegerType())])
input_df = spark.createDataFrame([1,4,5,7,9,10], schema= schema)

In [0]:
input_df.display()

In [0]:
refernce_df = spark.range(1,11).toDF('ID')
refernce_df.display()


In [0]:
refernce_df.join(input_df, refernce_df.ID == input_df.ID, 'leftanti').display()

## Q: frequency of each word in a string

In [0]:
sent = "Jingle bells, jingle bells Jingle all the way Oh what fun it is to ride In a one horse open sleigh"
sent = sent.lower()
dict = {}
for word in sent.split(" "):
    dict[word] = dict.get(word, 0) + 1
print(dict)

## SQL Query Differrences Question

In [0]:
df = spark.createDataFrame([(1, 'Akash', 50000), (2, 'Neha', 60000), (3, 'Priya', 70000), (4, 'Ravi', 80000), (5, 'Aarav', 62000), (2, 'Neha', 65000), (3, 'Priya', 75000)], ['id', 'name', 'salary'])

In [0]:
df.display()

In [0]:
df.createOrReplaceTempView("emp")


In [0]:
spark.sql("select count(*) from emp").show()
spark.sql("select count(1) from emp").show()
spark.sql("select count(distinct 2) from emp").show()
spark.sql("select 1 from emp").show()
spark.sql("select 1").show()


In [0]:
spark.sql("select DISTINCT salary from emp order by salary desc limit 1 offset 0").show()

**Q: Read a paruqet file stored in a datalake, read this file as a dataframe, then remove duplicate records, write back to datalake**

In [0]:
# read parqut data
df = spark.read.format('parquet')\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/workspace/default/rawData/salary.parquet")
df.display()

In [0]:
# add duplicate rows to the df
df.createOrReplaceTempView('salary')
spark.sql("select * from salary").show()

In [0]:
spark.sql("insert into salary select * from salary")


In [0]:
spark.sql("select * from salary").show()

In [0]:
# remove duplicate records from dataframe
df_with_duplicates = spark.sql("select * from salary")
df_without_duplicates = df_with_duplicates.dropDuplicates()

In [0]:
df_without_duplicates.display()

In [0]:
df_without_duplicates.write.format('parquet').save('/Volumes/workspace/default/rawData/salary1.parquet')

**Q Convert this table into this format**

In [0]:
data1 = [('a', 'aa', 1), ('a', 'aa', 2), ('b', 'bb', 3), ('b', 'bb', 4)]
data1_df = spark.createDataFrame(data1, ['col1', 'col2', 'col3'])
data1_df.display()

**Deexplode this dataframe**

In [0]:
data1_mod = data1_df.groupBy(['col1', 'col2']).agg(collect_list('col3').alias('col3'))
data1_mod.display()

In [0]:
# now explode again to make like prev one
data1_mod_exploded = data1_mod.select(col("col1"), col("col2"), explode(col("col3")).alias("col3"))
data1_mod_exploded.display()

**SQL JOins**

In [0]:
%sql
create table default.tableA(
  id int,
  value int
)


In [0]:
%sql
insert into default.tableA
values(1, 1), (1,2), (1,2), (0,0), (0,2), (null, 1), (null, null), (2,3)

In [0]:
%sql
select * from default.tableA

In [0]:
%sql
--tableB

create table default.tableB(
  id int,
  value int
);

insert into default.tableB values (1, 1), (1, 0), (0, 2), (null, null), (null, 1)


In [0]:
%sql
select * from default.tableA;

In [0]:
%sql
select * from default.tableB;

In [0]:
%sql
--inner join

select a.*, b.* from default.tableA a inner join default.tableB b on a.id = b.id;

In [0]:
%sql
--left join
select * from default.tableA a left join default.tableB b on a.id = b.id;
    
--right join

In [0]:
%sql
--outer
select * from default.tableA a full outer join default.tableB b on a.id = b.id;
    
